# Simplex Modeling API Demo

This notebook illustrates the higher-level modeling wrapper exposed by the local `simplex` module.

It focuses on the new API style:

- create variables by name with `addvar()` / `addVar()`
- build linear expressions with normal Python operators
- add constraints like `x <= 3` or `x + 2*y == 5`
- solve and inspect results by variable or by name


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

CANDIDATE_BUILD_DIRS = [
    Path('/data/solver/simplex/build-local'),
    Path('/data/solver/simplex/build'),
]

BUILD_DIR = next(
    (
        path
        for path in CANDIDATE_BUILD_DIRS
        if path.exists() and any(path.glob('simplex*.so'))
    ),
    None,
)

if BUILD_DIR is None:
    raise FileNotFoundError('Could not find a built simplex extension in build-local/ or build/.')

if str(BUILD_DIR) not in sys.path:
    sys.path.insert(0, str(BUILD_DIR))

import simplex

print('build dir:', BUILD_DIR)
print('simplex module:', simplex.__file__)


build dir: /data/solver/simplex/build-local
simplex module: /data/solver/simplex/build-local/simplex.cpython-313-x86_64-linux-gnu.so


## Example 1: the syntax you asked for

This is the intended ergonomic layer on top of the raw matrix API.

In [2]:
model = simplex.Model()

x = model.addvar('x')
y = model.addvar('y')

c1 = model.addConstr(x <= 3, name='cap_x')
c2 = model.addConstr(y <= 2, name='cap_y')
c3 = model.addConstr(x + 2 * y <= 6, name='mix_cap')

model.maximize(x + y)
solution = model.solve()

print('status   :', simplex.status_to_string(solution.status))
print('objective:', solution.obj)
print('x        :', solution.value(x))
print('y        :', solution.value('y'))
print('all vars :', solution.values)
print('dual c1 :', c1.pi)
print('dual c2 :', c2.pi)
print('dual c3 :', c3.pi)


status   : optimal
objective: 4.5
x        : 3.0000000000000004
y        : 1.4999999999999996
all vars : {'y': 1.4999999999999996, 'x': 3.0000000000000004}
dual c1 : 0.5
dual c2 : 0.0
dual c3 : 0.5


## Example 2: named variables, bounds, equality constraints, and aliases

The wrapper also supports `addVar`, `setObjective`, and reading the primal vector directly.

In [4]:
blend = simplex.Model()

protein = blend.addVar('protein', lb=0.0, ub=4.0)
fiber = blend.addVar('fiber', lb=0.0)

balance = blend.addConstr(protein + fiber == 5, name='balance')
minimum_mix = blend.addConstr(2 * protein + fiber >= 6, name='minimum_mix')

blend.setObjective(3 + 2 * protein + fiber, sense='min')
blend_solution = blend.solve()

print('status   :', simplex.status_to_string(blend_solution.status))
print('objective:', blend_solution.objective)
print('protein  :', blend_solution.value(protein))
print('fiber    :', blend_solution.value('fiber'))
print('x vector :', blend_solution.x)
print('raw obj  :', blend_solution.raw.obj)
print('balance pi:', balance.pi)
print('minimum_mix pi:', minimum_mix.pi)


status   : optimal
objective: 9.0
protein  : 0.9999999999999972
fiber    : 4.000000000000005
x vector : [1. 4.]
raw obj  : 6.0
balance pi: 0.0
minimum_mix pi: 1.0


## Quick reference

Useful methods in the new wrapper:

- `simplex.Model(options=...)`
- `model.addvar(name=None, lb=0.0, ub=inf, obj=0.0)`
- `model.addVar(...)` as an alias
- `model.addConstr(expr <= rhs)`
- `model.addConstr(expr == rhs)`
- `model.addConstr(expr >= rhs)`
- `model.minimize(expr)` / `model.maximize(expr)`
- `model.setObjective(expr, sense='min' | 'max')`
- `constraint = model.addConstr(...)` followed by `constraint.pi` after `solve()`
- `solution.value(var)` or `solution.value('name')`
- `solution.values` for a `{name: value}` dictionary

The original low-level `RevisedSimplex.solve(A, b, c, l, u)` interface is still available when you want to work directly with matrices.